# Watermark Survey — Complete Experiment Pipeline
**Authors:** Zahran Yahia Khan · Sofia Cobo Navas  
**Framework:** MarkLLM (Pan et al., 2024)  
**Scope:** 3 algorithms (KGW, Unigram, EXP) × 3 model scales (OPT-125M, 1.3B, 2.7B) × 2 datasets (C4, WMT16), 100 samples/combo  
**Research question:** How does LLM watermarking behavior change across model scale?


In [ ]:
# =========================================
# 🔧 SETUP
# =========================================

USE_DRIVE = True

import os, sys

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/watermark_survey"
else:
    BASE_PATH = "/content/watermark_survey"

for d in ["generated_texts", "attack_outputs/word_deletion",
          "attack_outputs/synonym", "attack_outputs/paraphrase",
          "results/scores", "results/tables", "batch_api"]:
    os.makedirs(os.path.join(BASE_PATH, d), exist_ok=True)

REPO_PATH = "/content/MarkLLM"
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/THU-BPM/MarkLLM.git /content/MarkLLM

# Keep Colab's preinstalled torch/numpy; only pin what MarkLLM needs
!pip install -q -r /content/MarkLLM/requirements.txt
!pip install -q "scipy>=1.14,<1.16"
!pip install -q "transformers==4.41.2"

import numpy, scipy, torch, transformers
print(f"   numpy:        {numpy.__version__}")
print(f"   scipy:        {scipy.__version__}")
print(f"   torch:        {torch.__version__}")
print(f"   transformers: {transformers.__version__}")
# ⚠️  If numpy prints 2.x here, RESTART RUNTIME and re-run this cell.

if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)

print("✅ Setup complete")


In [ ]:
# =========================================
# ⚙️ CONFIGURATION
# =========================================

ALGORITHMS    = ["KGW", "Unigram", "EXP"]
MODELS        = ["facebook/opt-125m", "facebook/opt-1.3b", "facebook/opt-2.7b"]
DATASETS      = ["c4", "wmt16"]
# OPT-125M included for generation + quality; excluded from robustness attacks
# because its baseline TPR is too low to measure meaningful degradation.
ATTACK_MODELS = [m for m in MODELS if "125m" not in m]
ATTACKS       = ["word_deletion", "synonym", "paraphrase"]

EXPECTED_LINES = 100
SCORES_DIR     = os.path.join(BASE_PATH, "results", "scores")
TABLES_DIR     = os.path.join(BASE_PATH, "results", "tables")

ALGO_CONFIGS = {
    "KGW":     "/content/MarkLLM/config/KGW.json",
    "Unigram": "/content/MarkLLM/config/Unigram.json",
    "EXP":     "/content/MarkLLM/config/EXP.json",
}

DATASET_PATHS = {
    "c4":    "/content/MarkLLM/dataset/c4/processed_c4.json",
    "wmt16": "/content/MarkLLM/dataset/wmt16_de_en/validation.jsonl",
}

# Score polarity: True = higher score means watermarked (KGW/Unigram z-score)
#                 False = lower score means watermarked  (EXP p-value)
POLARITY = {"KGW": True, "Unigram": True, "EXP": False}

print("✅ Configuration ready")


In [ ]:
# =========================================
# 🤖 MODEL LOADER
# =========================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from types import SimpleNamespace

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Device: {device}")

def load_model(model_name):
    """Load tokenizer + model in fp16. Works for all three OPT sizes."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16,
    ).to(device)
    return SimpleNamespace(
        model=model, tokenizer=tokenizer,
        vocab_size=model.config.vocab_size, device=device,
        gen_kwargs={
            "max_new_tokens": 200, "do_sample": True,
            "temperature": 0.7, "top_k": 50, "top_p": 0.9,
        },
    )

print("✅ Model loader ready")


In [ ]:
# =========================================
# 💧 WATERMARK LOADER
# =========================================

from watermark.auto_watermark import AutoWatermark

def load_watermark(algorithm_name, transformers_config):
    return AutoWatermark.load(
        algorithm_name=algorithm_name,
        algorithm_config=ALGO_CONFIGS[algorithm_name],
        transformers_config=transformers_config,
    )

print("✅ Watermark loader ready")


In [ ]:
# =========================================
# 📚 DATASET LOADER
# =========================================

import json

def load_prompts(dataset_name, n=100):
    """Load n prompts from the specified dataset."""
    path = DATASET_PATHS[dataset_name]
    assert os.path.exists(path), f"Dataset not found: {path}"
    prompts = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            item = json.loads(line)
            if dataset_name == "c4":
                # First 30 tokens as prompt
                text = item.get("prompt", "")
                prompts.append(" ".join(text.split()[:30]))
            elif dataset_name == "wmt16":
                assert "de" in item, f"'de' key missing in WMT16 line {i}"
                prompts.append(item["de"])
    print(f"✅ Loaded {len(prompts)} prompts from {dataset_name}")
    return prompts

print("✅ Dataset loader ready")


---
## Phase 1 — Text Generation

Generate 100 watermarked + 100 unwatermarked samples for each of the **18 combos**  
(3 algorithms × 3 models × 2 datasets). Files saved to `generated_texts/`.  
Each run is resumable — `generate_combo` appends from the last completed line.


In [ ]:
# =========================================
# 🚀 GENERATION ENGINE
# =========================================

import gc

def generate_combo(algorithm_name, model_name, dataset_name, n=100, batch_size=25):
    """Generate watermarked + unwatermarked texts for one combo.
    Resumable: skips already-completed samples via line count.
    """
    safe_model = model_name.replace("/", "_")
    wm_path  = os.path.join(BASE_PATH, "generated_texts",
                            f"{algorithm_name}_{safe_model}_{dataset_name}_wm.jsonl")
    uwm_path = os.path.join(BASE_PATH, "generated_texts",
                            f"{algorithm_name}_{safe_model}_{dataset_name}_uwm.jsonl")

    def count_lines(p):
        if not os.path.exists(p): return 0
        with open(p) as f: return sum(1 for _ in f)

    wm_done = count_lines(wm_path)
    if wm_done >= n and count_lines(uwm_path) >= n:
        print(f"[SKIP] {algorithm_name}/{safe_model}/{dataset_name} — already complete.")
        return wm_path, uwm_path

    print(f"\n[GEN] {algorithm_name} | {model_name} | {dataset_name}")
    tc = load_model(model_name)
    wm = load_watermark(algorithm_name, tc)
    prompts = load_prompts(dataset_name, n)

    with open(wm_path, "a") as fwm, open(uwm_path, "a") as fuwm:
        for i, prompt in enumerate(prompts):
            if i < wm_done:
                continue
            fwm.write(json.dumps({"prompt": prompt,
                                  "text": wm.generate_watermarked_text(prompt)}) + "\n")
            fuwm.write(json.dumps({"prompt": prompt,
                                   "text": wm.generate_unwatermarked_text(prompt)}) + "\n")
            if (i + 1) % batch_size == 0:
                print(f"   [{i+1}/{n}]")

    print(f"   ✅ Done")
    del tc, wm; torch.cuda.empty_cache(); gc.collect()
    return wm_path, uwm_path

print("✅ Generation engine ready")


In [ ]:
# =========================================
# ▶️ FULL GENERATION SWEEP (18 combos)
# =========================================
# Outer = model (expensive load), inner = algo × dataset.
# Runtime: ~3h on a T4 GPU.

import time

t0 = time.time()
for model_name in MODELS:
    for algorithm_name in ALGORITHMS:
        for dataset_name in DATASETS:
            generate_combo(algorithm_name, model_name, dataset_name)

print(f"\n✅ Generation complete — {(time.time() - t0) / 3600:.1f}h total")


---
## Phase 2 — Clean Detection

Run the watermark detector on all generated texts (no attacks applied).  
Produces **36 score files** (18 combos × {wm, uwm}).


In [ ]:
# =========================================
# 🔍 DETECTION HARNESS
# =========================================
# Shared by clean detection (Phase 2) and post-attack detection (Phase 3).
# The watermark config paths must match Phase 1 exactly — key/seed agreement
# is required for valid detection.

os.makedirs(SCORES_DIR, exist_ok=True)

def _extract_score(det):
    """Normalize detect_watermark output across algorithm families.
    KGW/Unigram return a z-score (higher = watermarked).
    EXP returns a p-value   (lower  = watermarked).
    """
    if isinstance(det, dict):
        score = det.get("score", det.get("z_score", det.get("p_value", 0.0)))
        flag  = det.get("is_watermarked", det.get("prediction", False))
    else:
        flag, score = det[0], det[1]
    return float(score), bool(flag)


def detect_jsonl_file(watermark, in_path, out_path):
    """Run detection on every text in a JSONL file. Resumable via line count."""
    if not os.path.exists(in_path):
        print(f"❌ Missing: {in_path}"); return None
    done = sum(1 for _ in open(out_path)) if os.path.exists(out_path) else 0
    with open(in_path) as fin, open(out_path, "a") as fout:
        for i, line in enumerate(fin):
            if i < done: continue
            item = json.loads(line)
            score, flag = _extract_score(watermark.detect_watermark(item["text"]))
            fout.write(json.dumps({**item,
                                   "score": score,
                                   "is_watermarked": flag}) + "\n")
    print(f"  ✅ {os.path.basename(out_path)}")
    return out_path


def load_score_array(path):
    """Read the 'score' field from every line of a detection JSONL file."""
    return [json.loads(l)["score"] for l in open(path)]


print("✅ Detection harness ready")


In [ ]:
# =========================================
# 🩺 SANITY CHECK — all Phase 1 + Phase 2 outputs
# =========================================
# Run before any model-loading step. Checks that all files are present
# and complete (≥100 lines). Expected totals:
#   generated_texts/ : 36 files (18 combos × {wm, uwm})
#   results/scores/  : 72 files (36 clean + 36 attack)
#   results/robustness.json : 36 rows

import glob

gen_dir     = os.path.join(BASE_PATH, "generated_texts")
score_dir   = SCORES_DIR
robust_path = os.path.join(BASE_PATH, "results", "robustness.json")

for path in [gen_dir, score_dir]:
    assert os.path.exists(path), f"❌ {path} not found — Drive not mounted?"

# Build expected file sets
expected_gen    = set()
expected_scores = set()
for m in MODELS:
    sm = m.replace("/", "_")
    for a in ALGORITHMS:
        for d in DATASETS:
            combo = f"{a}_{sm}_{d}"
            expected_gen.add(f"{combo}_wm.jsonl")
            expected_gen.add(f"{combo}_uwm.jsonl")
            expected_scores.add(f"detect_{combo}_clean_wm.jsonl")
            expected_scores.add(f"detect_{combo}_clean_uwm.jsonl")
for m in ATTACK_MODELS:
    sm = m.replace("/", "_")
    for a in ALGORITHMS:
        for d in DATASETS:
            combo = f"{a}_{sm}_{d}"
            for atk in ATTACKS:
                expected_scores.add(f"detect_{combo}_{atk}_wm.jsonl")

found_gen    = set(os.path.basename(p) for p in glob.glob(os.path.join(gen_dir,   "*.jsonl")))
found_scores = set(os.path.basename(p) for p in glob.glob(os.path.join(score_dir, "*.jsonl")))

def report_diff(label, expected, found, folder):
    missing = expected - found
    print(f"📁 {label}: {len(found)}/{len(expected)} files")
    if missing:
        print(f"   ❌ MISSING ({len(missing)}):")
        for f in sorted(missing): print(f"      - {f}")
    short = [(fn, n) for fn in sorted(expected & found)
             if (n := sum(1 for _ in open(os.path.join(folder, fn)))) < EXPECTED_LINES]
    if short:
        print(f"   ⚠️  INCOMPLETE:")
        for fn, n in short: print(f"      - {fn}: {n} lines")
    return not missing and not short

ok_gen    = report_diff("generated_texts", expected_gen,    found_gen,    gen_dir)
ok_scores = report_diff("results/scores",  expected_scores, found_scores, score_dir)

print(f"\n📁 robustness.json")
robust_ok = False
if not os.path.exists(robust_path):
    print("   ❌ MISSING — run Phase 3 aggregation first")
else:
    rows = json.load(open(robust_path))
    exp  = len(ATTACK_MODELS) * len(ALGORITHMS) * len(DATASETS) * len(ATTACKS)
    robust_ok = (len(rows) == exp)
    print(f"   {'✅' if robust_ok else '⚠️ '} {len(rows)}/{exp} rows")

if ok_gen and ok_scores and robust_ok:
    print("\n✅ All outputs present — ready for analysis.")
else:
    print("\n🛑 Fix missing files before proceeding.")


In [ ]:
# =========================================
# 📐 METRICS FUNCTION
# =========================================
# TPR/F1 at 10% FPR (fixed operating point) and at best (max-F1) threshold.
# Handles both score polarities via the POLARITY lookup table.

import numpy as np

def _classify(score, threshold, higher_is_watermarked):
    return score > threshold if higher_is_watermarked else score < threshold

def _confusion(wm_scores, uwm_scores, threshold, higher_is_watermarked):
    tp = sum(1 for s in wm_scores  if _classify(s, threshold, higher_is_watermarked))
    fp = sum(1 for s in uwm_scores if _classify(s, threshold, higher_is_watermarked))
    fn = len(wm_scores) - tp
    return tp, fp, fn

def _f1(tp, fp, fn):
    if tp == 0: return 0.0
    p = tp / (tp + fp); r = tp / (tp + fn)
    return 2 * p * r / (p + r) if (p + r) else 0.0

def compute_metrics(wm_scores, uwm_scores, *, higher_is_watermarked=True, target_fpr=0.10):
    wm  = np.asarray(wm_scores,  dtype=float)
    uwm = np.asarray(uwm_scores, dtype=float)

    # Threshold pinned to target FPR
    thresh_fpr = float(np.quantile(uwm, 1 - target_fpr) if higher_is_watermarked
                       else np.quantile(uwm, target_fpr))
    tp, fp, fn = _confusion(wm, uwm, thresh_fpr, higher_is_watermarked)
    tpr_at_fpr = tp / len(wm) if len(wm) else 0.0

    # Best F1 by threshold sweep
    best_f1 = best_tpr = 0.0; best_thresh = None
    for t in np.unique(np.concatenate([wm, uwm])):
        tp, fp, fn = _confusion(wm, uwm, t, higher_is_watermarked)
        f1 = _f1(tp, fp, fn)
        if f1 > best_f1:
            best_f1 = f1; best_tpr = tp / len(wm) if len(wm) else 0.0
            best_thresh = float(t)

    return {
        "n_wm": int(len(wm)), "n_uwm": int(len(uwm)),
        "target_fpr":  target_fpr,     "thresh_fpr":  thresh_fpr,
        "tpr_at_fpr":  tpr_at_fpr,     "f1_at_fpr":   _f1(*_confusion(wm, uwm, thresh_fpr, higher_is_watermarked)),
        "thresh_best": best_thresh,     "tpr_at_best": best_tpr,
        "f1_at_best":  best_f1,
    }

print("✅ Metrics function ready")


In [ ]:
# =========================================
# ▶️ CLEAN DETECTION SWEEP (18 combos × {wm, uwm} = 36 files)
# =========================================
# Outer = model (expensive load), middle = algorithm, inner = dataset.

import time

t0 = time.time()
for model_name in MODELS:
    safe_model = model_name.replace("/", "_")
    print(f"\n{'='*60}\n[MODEL] {model_name}\n{'='*60}")
    tc = load_model(model_name)

    for algorithm_name in ALGORITHMS:
        wm_obj = load_watermark(algorithm_name, tc)
        print(f"\n  [ALGO] {algorithm_name}")
        for dataset_name in DATASETS:
            combo   = f"{algorithm_name}_{safe_model}_{dataset_name}"
            t1 = time.time()
            detect_jsonl_file(wm_obj,
                os.path.join(BASE_PATH, "generated_texts", f"{combo}_wm.jsonl"),
                os.path.join(SCORES_DIR, f"detect_{combo}_clean_wm.jsonl"))
            detect_jsonl_file(wm_obj,
                os.path.join(BASE_PATH, "generated_texts", f"{combo}_uwm.jsonl"),
                os.path.join(SCORES_DIR, f"detect_{combo}_clean_uwm.jsonl"))
            print(f"    [{dataset_name}] {time.time() - t1:.1f}s")
        del wm_obj; gc.collect()

    del tc; torch.cuda.empty_cache(); gc.collect()

print(f"\n✅ Clean detection complete — {time.time() - t0:.1f}s total")


---
## Phase 3 — Robustness Attacks

Applied to **ATTACK_MODELS** (OPT-1.3B and OPT-2.7B) only.  
OPT-125M is excluded because its baseline TPR is too low to measure meaningful degradation.

| Attack | Parameters |
|---|---|
| Word deletion | `ratio = 0.3` |
| Synonym substitution | `ratio = 0.5`, `bert-large-uncased` |
| Paraphrase | GPT-4o via OpenAI Batch API (results pre-generated) |


In [ ]:
# =========================================
# 🗑️ PHASE 3A — WORD DELETION (ratio=0.3)
# =========================================

import random, time, gc
from evaluation.tools.text_editor import WordDeletion

random.seed(42)
WD_DIR = os.path.join(BASE_PATH, "attack_outputs", "word_deletion")
os.makedirs(WD_DIR, exist_ok=True)
attacker = WordDeletion(ratio=0.3)

t0 = time.time()

# --- Step 1: apply attack ---
print("=== Step 1: Applying word deletion ===")
for model_name in ATTACK_MODELS:
    safe_model = model_name.replace("/", "_")
    for algorithm_name in ALGORITHMS:
        for dataset_name in DATASETS:
            combo    = f"{algorithm_name}_{safe_model}_{dataset_name}"
            in_path  = os.path.join(BASE_PATH, "generated_texts", f"{combo}_wm.jsonl")
            out_path = os.path.join(WD_DIR, f"{combo}.jsonl")
            done = sum(1 for _ in open(out_path)) if os.path.exists(out_path) else 0
            with open(in_path) as fin, open(out_path, "a") as fout:
                for i, line in enumerate(fin):
                    if i < done: continue
                    item = json.loads(line)
                    fout.write(json.dumps({"prompt": item.get("prompt", ""),
                                           "text": attacker.edit(item["text"], None),
                                           "original": item["text"],
                                           "line_idx": i}) + "\n")
            n = sum(1 for _ in open(out_path))
            print(f"  ✅ {combo}: {n} samples")

print(f"⏱  Attack: {time.time() - t0:.1f}s")

# --- Step 2: detect ---
print("\n=== Step 2: Detection ===")
t1 = time.time()
for model_name in ATTACK_MODELS:
    safe_model = model_name.replace("/", "_")
    tc = load_model(model_name)
    for algorithm_name in ALGORITHMS:
        wm_obj = load_watermark(algorithm_name, tc)
        for dataset_name in DATASETS:
            combo    = f"{algorithm_name}_{safe_model}_{dataset_name}"
            detect_jsonl_file(wm_obj,
                os.path.join(WD_DIR, f"{combo}.jsonl"),
                os.path.join(SCORES_DIR, f"detect_{combo}_word_deletion_wm.jsonl"))
        del wm_obj; gc.collect()
    del tc; torch.cuda.empty_cache(); gc.collect()

print(f"\n✅ Phase 3a complete — {time.time() - t1:.1f}s detection")


In [ ]:
# =========================================
# 🔄 PHASE 3B — SYNONYM SUBSTITUTION (ratio=0.5, bert-large-uncased)
# =========================================
# BERT stays on GPU for the full attack phase, then is freed before
# detection loads the OPT models.

import random, time, gc
from transformers import AutoTokenizer as HFAutoTokenizer, AutoModelForMaskedLM
from evaluation.tools.text_editor import ContextAwareSynonymSubstitution

random.seed(42)
SYN_DIR = os.path.join(BASE_PATH, "attack_outputs", "synonym")
os.makedirs(SYN_DIR, exist_ok=True)

t0 = time.time()

# --- Step 1: load BERT, apply attack ---
print("=== Step 1: Loading BERT and applying synonym substitution ===")
bert_tokenizer = HFAutoTokenizer.from_pretrained("bert-large-uncased")
bert_model     = AutoModelForMaskedLM.from_pretrained("bert-large-uncased").to(device)
bert_model.eval()
attacker = ContextAwareSynonymSubstitution(
    ratio=0.5, tokenizer=bert_tokenizer, model=bert_model, device=device,
)
print(f"BERT loaded on {device}\n")

for model_name in ATTACK_MODELS:
    safe_model = model_name.replace("/", "_")
    for algorithm_name in ALGORITHMS:
        for dataset_name in DATASETS:
            combo    = f"{algorithm_name}_{safe_model}_{dataset_name}"
            in_path  = os.path.join(BASE_PATH, "generated_texts", f"{combo}_wm.jsonl")
            out_path = os.path.join(SYN_DIR, f"{combo}.jsonl")
            done = sum(1 for _ in open(out_path)) if os.path.exists(out_path) else 0
            t1 = time.time()
            with open(in_path) as fin, open(out_path, "a") as fout:
                for i, line in enumerate(fin):
                    if i < done: continue
                    item = json.loads(line)
                    fout.write(json.dumps({"prompt": item.get("prompt", ""),
                                           "text": attacker.edit(item["text"], None),
                                           "original": item["text"],
                                           "line_idx": i}) + "\n")
            n = sum(1 for _ in open(out_path))
            print(f"  ✅ {combo}: {n} samples ({time.time()-t1:.1f}s)")

print(f"\n⏱  Attack phase: {(time.time()-t0)/60:.1f} min")

# Free BERT before loading OPT
del attacker, bert_model, bert_tokenizer
torch.cuda.empty_cache(); gc.collect()
print("🧹 BERT freed")

# --- Step 2: detect ---
print("\n=== Step 2: Detection ===")
t2 = time.time()
for model_name in ATTACK_MODELS:
    safe_model = model_name.replace("/", "_")
    tc = load_model(model_name)
    for algorithm_name in ALGORITHMS:
        wm_obj = load_watermark(algorithm_name, tc)
        for dataset_name in DATASETS:
            combo    = f"{algorithm_name}_{safe_model}_{dataset_name}"
            detect_jsonl_file(wm_obj,
                os.path.join(SYN_DIR, f"{combo}.jsonl"),
                os.path.join(SCORES_DIR, f"detect_{combo}_synonym_wm.jsonl"))
        del wm_obj; gc.collect()
    del tc; torch.cuda.empty_cache(); gc.collect()

print(f"\n✅ Phase 3b complete — {(time.time()-t0)/60:.1f} min total")


### Phase 3c — Paraphrase Attack

Paraphrased texts were generated offline via the **OpenAI Batch API** (GPT-4o).  
Before running the cell below, the batch results file must already be on Drive at:

```
batch_api/paraphrase_batch_v2_results.jsonl
```

`custom_id` format per row: `{combo}_{NNN}` (e.g. `KGW_facebook_opt-1.3b_c4_042`).


In [ ]:
# =========================================
# 📝 PHASE 3C — PARAPHRASE SWEEP + DETECTION
# =========================================

import time, gc
from collections import defaultdict

PARA_DIR           = os.path.join(BASE_PATH, "attack_outputs", "paraphrase")
batch_results_path = os.path.join(BASE_PATH, "batch_api", "paraphrase_batch_v2_results.jsonl")
os.makedirs(PARA_DIR, exist_ok=True)
assert os.path.exists(batch_results_path), f"Batch results not found: {batch_results_path}"

# --- Step 1: parse batch results into per-combo JSONL files ---
print("=== Step 1: Parsing batch results ===")
combo_buckets = defaultdict(dict)
with open(batch_results_path) as f:
    for line in f:
        row   = json.loads(line)
        cid   = row["custom_id"]
        combo = "_".join(cid.split("_")[:-1])
        idx   = int(cid.split("_")[-1])
        text  = row["response"]["body"]["choices"][0]["message"]["content"]
        combo_buckets[combo][idx] = text

t0 = time.time()
for combo in sorted(combo_buckets):
    src_path = os.path.join(BASE_PATH, "generated_texts", f"{combo}_wm.jsonl")
    out_path = os.path.join(PARA_DIR, f"{combo}.jsonl")
    n_missing = 0
    with open(src_path) as fin, open(out_path, "w") as fout:
        for i, line in enumerate(fin):
            item = json.loads(line)
            if i not in combo_buckets[combo]:
                n_missing += 1; continue
            fout.write(json.dumps({"prompt": item.get("prompt", ""),
                                   "text": combo_buckets[combo][i],
                                   "original": item["text"],
                                   "line_idx": i}) + "\n")
    n = sum(1 for _ in open(out_path))
    flag = "⚠️ " if n_missing else "✅"
    print(f"  {flag} {combo}: {n} written" + (f" ({n_missing} missing)" if n_missing else ""))

print(f"⏱  Parse: {time.time()-t0:.1f}s")

# --- Step 2: detect ---
print("\n=== Step 2: Detection ===")
t1 = time.time()
for model_name in ATTACK_MODELS:
    safe_model = model_name.replace("/", "_")
    tc = load_model(model_name)
    for algorithm_name in ALGORITHMS:
        wm_obj = load_watermark(algorithm_name, tc)
        for dataset_name in DATASETS:
            combo    = f"{algorithm_name}_{safe_model}_{dataset_name}"
            detect_jsonl_file(wm_obj,
                os.path.join(PARA_DIR, f"{combo}.jsonl"),
                os.path.join(SCORES_DIR, f"detect_{combo}_paraphrase_wm.jsonl"))
        del wm_obj; gc.collect()
    del tc; torch.cuda.empty_cache(); gc.collect()

print(f"\n✅ Phase 3c complete — {(time.time()-t1)/60:.1f} min")


In [ ]:
# =========================================
# 📊 ROBUSTNESS AGGREGATION
# =========================================
# For each (combo × attack), compute post-attack metrics.
# Negative class = clean_uwm (not attacked unwatermarked text):
# FPR is calibrated against natural text, the realistic deployment scenario.
# Writes results/robustness.json — 36 rows.

rows = []

for model_name in ATTACK_MODELS:
    safe_model = model_name.replace("/", "_")
    for algorithm_name in ALGORITHMS:
        for dataset_name in DATASETS:
            combo     = f"{algorithm_name}_{safe_model}_{dataset_name}"
            clean_wm  = load_score_array(os.path.join(SCORES_DIR, f"detect_{combo}_clean_wm.jsonl"))
            clean_uwm = load_score_array(os.path.join(SCORES_DIR, f"detect_{combo}_clean_uwm.jsonl"))
            clean_m   = compute_metrics(clean_wm, clean_uwm,
                                        higher_is_watermarked=POLARITY[algorithm_name])
            for attack in ATTACKS:
                attack_path = os.path.join(SCORES_DIR, f"detect_{combo}_{attack}_wm.jsonl")
                if not os.path.exists(attack_path):
                    print(f"  ⚠️  missing: {attack_path}"); continue
                attack_m = compute_metrics(
                    load_score_array(attack_path), clean_uwm,
                    higher_is_watermarked=POLARITY[algorithm_name])
                rows.append({
                    "algorithm": algorithm_name, "model": model_name,
                    "dataset": dataset_name,     "attack": attack,
                    "clean": clean_m,             "attack_metrics": attack_m,
                    "deltas": {k: clean_m[k] - attack_m[k]
                               for k in ["tpr_at_fpr", "f1_at_fpr",
                                         "tpr_at_best", "f1_at_best"]},
                })

out_path = os.path.join(BASE_PATH, "results", "robustness.json")
with open(out_path, "w") as f:
    json.dump(rows, f, indent=2)
print(f"✅ Wrote {len(rows)} rows to {out_path}")

# Quick summary
print("\n=== Average TPR@10%FPR drop by attack ===")
for attack in ATTACKS:
    drops = [r["deltas"]["tpr_at_fpr"] for r in rows if r["attack"] == attack]
    print(f"  {attack:<15} avg drop = {sum(drops)/len(drops):+.4f}")

print("\n=== Paraphrase drop by algorithm ===")
for algo in ALGORITHMS:
    drops = [r["deltas"]["tpr_at_fpr"] for r in rows
             if r["attack"] == "paraphrase" and r["algorithm"] == algo]
    if drops:
        print(f"  {algo:<10} avg drop = {sum(drops)/len(drops):+.4f}")


---
## Phase 4 — Text Quality

Two complementary metrics, both CPU-only after GPT-2 is freed.

### Perplexity (PPL)
GPT-2 (small) as external scorer, matching MarkLLM Table 4.  
**Reported for C4 only.** WMT16 PPL values are omitted: GPT-2 (an English-trained model) assigns high perplexity to fluent German continuations on grounds of language mismatch, not watermark cost. Including them would conflate the two.

> **Unigram ΔPPL note:** Unigram produces a *negative* ΔPPL on C4 (watermarked PPL < unwatermarked PPL). This is a scorer-prior artifact — Unigram's fixed green list overlaps with GPT-2's high-probability tokens, biasing the external scorer in Unigram's favor. The watermarked text is not visibly higher quality. Interpret with this interaction in mind.

### Log-Diversity
Welleck-style n-gram diversity (n = 2, 3, 4), reported at two granularities:
- **Corpus-level:** pools n-grams across all 100 texts — captures cross-text variety.
- **Per-text:** computes log-div per sample, then averages — captures within-text repetition.

Both directions agree (wm < uwm across most combos) but answer different questions. Absolute values differ from MarkLLM Table 4 due to aggregation differences; within-study Δ values are valid for comparison.

> **OPT-125M WMT16 note:** Log-diversity values for OPT-125M on WMT16 are inflated by partially-degenerate output (verbatim prompt repetition + topic drift). N-gram diversity rewards novelty without measuring coherence. Comparative claims across scale should focus on the **1.3B → 2.7B** transition.


In [ ]:
# =========================================
# 📏 PPL SCORER + SWEEP + AGGREGATION
# =========================================

import statistics, time, gc
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

ppl_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
ppl_model     = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ppl_model.eval()
print(f"GPT-2 loaded on {device}")

def compute_ppl(text):
    """Per-sample perplexity via GPT-2 NLL. Returns None for unscorable text."""
    if not text or not text.strip(): return None
    inputs = ppl_tokenizer(text, return_tensors="pt",
                           truncation=True, max_length=1024).to(device)
    if inputs.input_ids.shape[1] < 2: return None
    with torch.no_grad():
        loss = ppl_model(**inputs, labels=inputs.input_ids).loss
    return float(torch.exp(loss).item())

def compute_ppl_jsonl_file(in_path, out_path):
    """Run PPL on every text in a JSONL file. Resumable."""
    if not os.path.exists(in_path):
        print(f"  ❌ Missing: {in_path}"); return
    done = sum(1 for _ in open(out_path)) if os.path.exists(out_path) else 0
    with open(in_path) as fin, open(out_path, "a") as fout:
        for i, line in enumerate(fin):
            if i < done: continue
            item = json.loads(line)
            fout.write(json.dumps({**item, "ppl": compute_ppl(item["text"])}) + "\n")

# --- Sweep (36 files) ---
print("\n=== PPL sweep (36 files) ===")
t0 = time.time(); total = 0
for model_name in MODELS:
    safe_model = model_name.replace("/", "_")
    for algo in ALGORITHMS:
        for ds in DATASETS:
            combo = f"{algo}_{safe_model}_{ds}"
            for variant in ["wm", "uwm"]:
                compute_ppl_jsonl_file(
                    os.path.join(BASE_PATH, "generated_texts", f"{combo}_{variant}.jsonl"),
                    os.path.join(SCORES_DIR,                   f"ppl_{combo}_{variant}.jsonl"))
                total += 1
                print(f"  [{total:2d}/36] {combo}_{variant}")

print(f"\n✅ PPL sweep done — {(time.time()-t0)/60:.1f} min")

# --- Aggregate ---
print("\n=== Aggregation ===")
ppl_summary = {}
for model_name in MODELS:
    safe_model = model_name.replace("/", "_")
    for algo in ALGORITHMS:
        for ds in DATASETS:
            combo = f"{algo}_{safe_model}_{ds}"
            entry = {}
            for variant in ["wm", "uwm"]:
                ppls  = [json.loads(l)["ppl"]
                         for l in open(os.path.join(SCORES_DIR, f"ppl_{combo}_{variant}.jsonl"))]
                valid = [p for p in ppls if p is not None]
                entry[variant] = {"n": len(valid),
                                  "mean": statistics.mean(valid) if valid else None,
                                  "median": statistics.median(valid) if valid else None}
            entry["delta_mean"] = entry["wm"]["mean"] - entry["uwm"]["mean"]
            ppl_summary[combo] = entry

ppl_path = os.path.join(BASE_PATH, "results", "quality_ppl.json")
with open(ppl_path, "w") as f:
    json.dump(ppl_summary, f, indent=2)
print(f"✅ Written to {ppl_path}")

# Free GPT-2
del ppl_model, ppl_tokenizer
gc.collect(); torch.cuda.empty_cache()
print("🧹 GPT-2 freed")


In [ ]:
# =========================================
# 🌱 LOG-DIVERSITY — corpus-level + per-text
# =========================================
# CPU-only. Both aggregations run in a single pass over the 18 combos.

import math, statistics

def compute_log_diversity(texts):
    """Corpus-level log-diversity, n=2,3,4. Pools n-grams across all texts."""
    score = 0.0
    for n in [2, 3, 4]:
        ngrams = []
        for text in texts:
            toks = text.split()
            ngrams.extend(tuple(toks[i:i+n]) for i in range(len(toks) - n + 1))
        if not ngrams: continue
        rep_n = 1 - len(set(ngrams)) / len(ngrams)
        if rep_n < 1.0: score -= math.log(1 - rep_n)
    return score

def per_text_log_div(text):
    """Single-text log-diversity, n=2,3,4. None if text too short."""
    toks = text.split()
    if len(toks) < 4: return None
    score = 0.0
    for n in [2, 3, 4]:
        ngrams = [tuple(toks[i:i+n]) for i in range(len(toks) - n + 1)]
        if not ngrams: continue
        rep_n = 1 - len(set(ngrams)) / len(ngrams)
        if rep_n < 1.0: score -= math.log(1 - rep_n)
    return score

logdiv_corpus  = {}
logdiv_pertext = {}

for model_name in MODELS:
    safe_model = model_name.replace("/", "_")
    for algo in ALGORITHMS:
        for ds in DATASETS:
            combo = f"{algo}_{safe_model}_{ds}"
            entry_c = {}; entry_p = {}
            for variant in ["wm", "uwm"]:
                path  = os.path.join(BASE_PATH, "generated_texts", f"{combo}_{variant}.jsonl")
                texts = [json.loads(l)["text"] for l in open(path)]
                # Corpus-level
                entry_c[variant] = {"n": len(texts),
                                    "log_diversity": compute_log_diversity(texts)}
                # Per-text
                pt_scores = [s for t in texts if (s := per_text_log_div(t)) is not None]
                entry_p[variant] = {"n": len(pt_scores),
                                    "mean_log_diversity": (statistics.mean(pt_scores)
                                                           if pt_scores else None)}
            entry_c["delta"] = (entry_c["wm"]["log_diversity"]
                                - entry_c["uwm"]["log_diversity"])
            entry_p["delta"] = (entry_p["wm"]["mean_log_diversity"]
                                - entry_p["uwm"]["mean_log_diversity"])
            logdiv_corpus[combo]  = entry_c
            logdiv_pertext[combo] = entry_p
            print(f"  {combo}  Δ_corpus={entry_c['delta']:+.3f}  Δ_pertext={entry_p['delta']:+.3f}")

with open(os.path.join(BASE_PATH, "results", "quality_log_diversity.json"), "w") as f:
    json.dump(logdiv_corpus, f, indent=2)
with open(os.path.join(BASE_PATH, "results", "quality_log_diversity_per_text.json"), "w") as f:
    json.dump(logdiv_pertext, f, indent=2)

print("\n✅ Log-diversity written to results/")


---
## Phase 5 — Results

Compile the three result tables and print the RQ-organized analysis.  
All cells below are pure CPU and pandas — no model loading required.


In [ ]:
# =========================================
# 📊 FINAL TABLES — detectability · robustness · quality
# =========================================
# Writes CSV, Markdown, and LaTeX versions of all three tables.

import pandas as pd

os.makedirs(TABLES_DIR, exist_ok=True)

def save_table(df, name):
    base = os.path.join(TABLES_DIR, name)
    df.to_csv(base + ".csv", index=False)
    with open(base + ".md",  "w") as f: f.write(df.to_markdown(index=False, floatfmt=".3f"))
    with open(base + ".tex", "w") as f: f.write(df.to_latex(index=False, float_format="%.3f"))
    print(f"  ✅ {name}: {df.shape[0]} rows → .{{csv,md,tex}}")

def short_model(name):
    return name.split("/")[-1].replace("opt-", "").upper()

# --- Table 1: Detectability (18 combos) ---
det_rows = []
for model_name in MODELS:
    sm = model_name.replace("/", "_")
    for algo in ALGORITHMS:
        for ds in DATASETS:
            combo = f"{algo}_{sm}_{ds}"
            m = compute_metrics(
                load_score_array(os.path.join(SCORES_DIR, f"detect_{combo}_clean_wm.jsonl")),
                load_score_array(os.path.join(SCORES_DIR, f"detect_{combo}_clean_uwm.jsonl")),
                higher_is_watermarked=POLARITY[algo],
            )
            det_rows.append({"Algorithm": algo, "Model": short_model(model_name),
                             "Dataset": ds,
                             "TPR@10%FPR": m["tpr_at_fpr"], "F1@10%FPR": m["f1_at_fpr"],
                             "TPR@best":   m["tpr_at_best"], "F1@best":   m["f1_at_best"]})
det_df = pd.DataFrame(det_rows)
print("📊 Table 1 — Detectability")
save_table(det_df, "detectability")

# --- Table 2: Robustness (12 eligible combos, pivoted wide) ---
robust_rows = json.load(open(os.path.join(BASE_PATH, "results", "robustness.json")))
robust_idx  = {}
for r in robust_rows:
    key = (r["algorithm"], r["model"], r["dataset"])
    if key not in robust_idx:
        robust_idx[key] = {"Algorithm": r["algorithm"], "Model": short_model(r["model"]),
                           "Dataset": r["dataset"],
                           "NoAttack_TPR": r["clean"]["tpr_at_fpr"],
                           "NoAttack_F1":  r["clean"]["f1_at_fpr"]}
    atk = r["attack"]
    robust_idx[key][f"{atk}_TPR"] = r["attack_metrics"]["tpr_at_fpr"]
    robust_idx[key][f"{atk}_F1"]  = r["attack_metrics"]["f1_at_fpr"]
robust_df = pd.DataFrame(list(robust_idx.values())).sort_values(
    ["Algorithm", "Model", "Dataset"]).reset_index(drop=True)
print("\n📊 Table 2 — Robustness")
save_table(robust_df, "robustness")

# --- Table 3: Quality (18 combos) ---
ppl_data    = json.load(open(os.path.join(BASE_PATH, "results", "quality_ppl.json")))
logdiv_corp = json.load(open(os.path.join(BASE_PATH, "results", "quality_log_diversity.json")))
logdiv_pt   = json.load(open(os.path.join(BASE_PATH, "results", "quality_log_diversity_per_text.json")))

qual_rows = []
for model_name in MODELS:
    sm = model_name.replace("/", "_")
    for algo in ALGORITHMS:
        for ds in DATASETS:
            combo  = f"{algo}_{sm}_{ds}"
            ppl_e  = ppl_data[combo]
            corp_e = logdiv_corp[combo]; pt_e = logdiv_pt[combo]
            # PPL: C4 only — WMT16 omitted (English scorer vs German output)
            qual_rows.append({
                "Algorithm": algo, "Model": short_model(model_name), "Dataset": ds,
                "PPL_wm":  ppl_e["wm"]["mean"]  if ds == "c4" else None,
                "PPL_uwm": ppl_e["uwm"]["mean"] if ds == "c4" else None,
                "ΔPPL":    ppl_e["delta_mean"]  if ds == "c4" else None,
                "LogDiv_corpus_wm":    corp_e["wm"]["log_diversity"],
                "LogDiv_corpus_uwm":   corp_e["uwm"]["log_diversity"],
                "ΔLogDiv_corpus":      corp_e["delta"],
                "LogDiv_pertext_wm":   pt_e["wm"]["mean_log_diversity"],
                "LogDiv_pertext_uwm":  pt_e["uwm"]["mean_log_diversity"],
                "ΔLogDiv_pertext":     pt_e["delta"],
            })
qual_df = pd.DataFrame(qual_rows)
print("\n📊 Table 3 — Quality")
save_table(qual_df, "quality")

print(f"\n✅ All tables saved to {TABLES_DIR}")


In [ ]:
# =========================================
# 🔬 RQ-ORGANIZED ANALYSIS
# =========================================

det_df      = pd.read_csv(os.path.join(TABLES_DIR, "detectability.csv"))
robust_df   = pd.read_csv(os.path.join(TABLES_DIR, "robustness.csv"))
qual_df     = pd.read_csv(os.path.join(TABLES_DIR, "quality.csv"))
robust_json = json.load(open(os.path.join(BASE_PATH, "results", "robustness.json")))

# ── RQ1: Does detectability depend on model scale? ─────────────────────
print("=" * 72)
print("RQ1 — DETECTABILITY ACROSS SCALES (TPR @ 10% FPR)")
print("=" * 72)
for metric, label in [("TPR@10%FPR", "TPR"), ("F1@10%FPR", "F1")]:
    print(f"\n  {label}:")
    print(det_df.pivot_table(index=["Algorithm", "Dataset"],
                             columns="Model", values=metric
          ).reindex(columns=["125M", "1.3B", "2.7B"]
          ).to_string(float_format=lambda x: f"{x:.3f}"))

# ── RQ2: Does robustness change with scale? ────────────────────────────
for attack in ATTACKS:
    col = f"{attack}_TPR"
    if col not in robust_df.columns: continue
    print("\n" + "=" * 72)
    print(f"RQ2 — POST-{attack.upper()} TPR @ 10% FPR")
    print("=" * 72)
    print(robust_df.pivot_table(index=["Algorithm", "Dataset"],
                                columns="Model", values=col
          ).reindex(columns=["1.3B", "2.7B"]
          ).to_string(float_format=lambda x: f"{x:.3f}"))

print("\n" + "=" * 72)
print("RQ2 — PARAPHRASE TPR DROP BY ALGORITHM")
print("=" * 72)
print(f"  {'Algorithm':<12}{'Clean TPR':>12}{'Para TPR':>12}{'Δ (drop)':>12}")
print("  " + "-" * 48)
for algo in ALGORITHMS:
    clean = [r["clean"]["tpr_at_fpr"] for r in robust_json
             if r["algorithm"] == algo and r["attack"] == "paraphrase"]
    para  = [r["attack_metrics"]["tpr_at_fpr"] for r in robust_json
             if r["algorithm"] == algo and r["attack"] == "paraphrase"]
    if clean:
        c, p = sum(clean)/len(clean), sum(para)/len(para)
        print(f"  {algo:<12}{c:>12.3f}{p:>12.3f}{c-p:>+12.3f}")

# ── RQ3: Does quality cost shrink at larger scale? ─────────────────────
print("\n" + "=" * 72)
print("RQ3 — ΔPPL ACROSS SCALES (C4 only; WMT16 omitted)")
print("=" * 72)
print(qual_df[qual_df["Dataset"] == "c4"].pivot_table(
    index="Algorithm", columns="Model", values="ΔPPL",
).reindex(columns=["125M", "1.3B", "2.7B"]
).to_string(float_format=lambda x: f"{x:+.3f}"))

for label, col in [("Δ LOG-DIVERSITY (corpus-level)", "ΔLogDiv_corpus"),
                   ("Δ LOG-DIVERSITY (per-text)",     "ΔLogDiv_pertext")]:
    print("\n" + "=" * 72)
    print(f"RQ3 — {label}")
    print("=" * 72)
    for ds in DATASETS:
        print(f"\n  Dataset: {ds}")
        print(qual_df[qual_df["Dataset"] == ds].pivot_table(
            index="Algorithm", columns="Model", values=col,
        ).reindex(columns=["125M", "1.3B", "2.7B"]
        ).to_string(float_format=lambda x: f"{x:+.3f}"))

# ── Cross-cutting: C4 vs WMT16 ─────────────────────────────────────────
print("\n" + "=" * 72)
print("CROSS-CUTTING — DATASET EFFECT (TPR @ 10% FPR, 1.3B + 2.7B only)")
print("=" * 72)
dom = det_df[det_df["Model"].isin(["1.3B", "2.7B"])].pivot_table(
    index=["Algorithm", "Model"], columns="Dataset", values="TPR@10%FPR",
)
dom["Δ (wmt16 - c4)"] = dom["wmt16"] - dom["c4"]
print(dom.to_string(float_format=lambda x: f"{x:+.3f}"))
